# CAST Phase 2 — Cultural Prompt Ablation

This notebook evaluates Gemma 4 31B under four zero-shot prompt
conditions (P0–P3) for the nine BRIGHTER languages. It uses one
canonical P0 baseline, stores per-sample predictions, calculates
English macro-F1 over its five active emotions, and exports the
result tables used by the later phases.

**Execution order:** setup → data → metrics → prompts → model/inference
(when enabled) → validation → exports.


In [ ]:
# Environment setup
!pip install -q -U datasets bitsandbytes accelerate huggingface_hub

from google.colab import drive
drive.mount("/content/drive", force_remount=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 48.1 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# Configuration
import json
import os
import warnings

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from sklearn.metrics import f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SAVE_DIR = "/content/drive/MyDrive/EmotionDetection/CAST_checkpoints"
P2_SAVE_DIR = f"{SAVE_DIR}/Phase2"
PRED_DIR = f"{P2_SAVE_DIR}/predictions"
CACHE_DIR = f"{SAVE_DIR}/data_cache"
MODEL_CACHE_DIR = f"{SAVE_DIR}/model_cache"
LOCAL_MODEL_CACHE = "/content/hf_cache_local"

for directory in [
    P2_SAVE_DIR,
    PRED_DIR,
    CACHE_DIR,
    MODEL_CACHE_DIR,
    LOCAL_MODEL_CACHE,
]:
    os.makedirs(directory, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{MODEL_CACHE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{MODEL_CACHE_DIR}/datasets"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

EMOTION_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LANG_ORDER = ["eng", "hin", "rus", "hau", "kin", "sun", "yor", "vmw", "pcm"]
LANG_CODES = LANG_ORDER.copy()
PROMPT_KEYS = ["p0", "p1", "p2", "p3"]
CANONICAL_P0_KEY = "p0"
DATASET_HF_NAME = "brighter-dataset/BRIGHTER-emotion-categories"

LANGUAGES = {
    "eng": {"name": "English", "tier": 1, "subfamily": "Germanic"},
    "hin": {"name": "Hindi", "tier": 1, "subfamily": "Indo-Aryan"},
    "rus": {"name": "Russian", "tier": 1, "subfamily": "Slavic"},
    "hau": {"name": "Hausa", "tier": 2, "subfamily": "Chadic"},
    "kin": {"name": "Kinyarwanda", "tier": 2, "subfamily": "Bantu (Great Lakes)"},
    "sun": {"name": "Sundanese", "tier": 2, "subfamily": "Sundic"},
    "yor": {"name": "Yoruba", "tier": 3, "subfamily": "Yoruboid"},
    "vmw": {"name": "Emakhuwa", "tier": 3, "subfamily": "Bantu (Makua-Lomwe)"},
    "pcm": {"name": "Nigerian Pidgin", "tier": 3, "subfamily": "English-Based"},
}
if list(LANGUAGES) != LANG_ORDER:
    raise ValueError("LANGUAGES must follow LANG_ORDER.")

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

GEMMA_HF_NAME = "unsloth/gemma-4-31B-it-unsloth-bnb-4bit"
LOAD_IN_4BIT_ON_THE_FLY = False
GENERATION_CONFIG = {
    "max_new_tokens": 30,
    "do_sample": False,
}

# False validates and exports the completed run. Set True only to generate
# genuinely missing or invalid language/prompt predictions.
RUN_MODEL_INFERENCE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Model: {GEMMA_HF_NAME}")
print(f"Outputs: {P2_SAVE_DIR}")


# ── Phase 2 annotation constants ─────────────────────────────────────────────
PHASE2_SCOPE = "Internal Gemma prompt comparison"
PROMPT_LABELS = {
    "p0": "P0 macro-F1 (basic prompt)",
    "p1": "P1 macro-F1 (cultural identity context)",
    "p2": "P2 macro-F1 (cultural examples)",
    "p3": "P3 macro-F1 (combined cultural context)",
}

Device: cuda
Model: unsloth/gemma-4-31B-it-unsloth-bnb-4bit
Outputs: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase2


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────

def load_split(lang_code: str, split: str):
    cpath = f"{CACHE_DIR}/{lang_code}_{split}.parquet"
    if os.path.exists(cpath):
        return pd.read_parquet(cpath)
    try:
        ds = load_dataset(DATASET_HF_NAME, lang_code)
        split_key = split if split in ds else {"validation": "dev", "dev": "validation"}.get(split)
        if not split_key or split_key not in ds:
            return None
        df = ds[split_key].to_pandas()
        for e in EMOTION_ORDER:
            if e not in df.columns:
                df[e] = 0
        df.to_parquet(cpath)
        return df
    except Exception as exc:
        print(f"  Could not load {lang_code}/{split}: {exc}")
        return None

def load_language(lang_code: str) -> dict:
    splits = {}
    for split in ["train", "test", "validation"]:
        df = load_split(lang_code, split)
        if df is not None:
            splits[split if split != "validation" else "validation"] = df
    if "validation" not in splits:
        raise KeyError(f"No dev/validation split for {lang_code}")
    return splits


def get_emotion_cols(df: pd.DataFrame) -> list:
    return [e for e in EMOTION_ORDER if e in df.columns and df[e].fillna(0).sum() > 0]


print("Loading BRIGHTER for all 9 languages ...")
DATA: dict = {}
for code in LANG_CODES:
    try:
        DATA[code] = load_language(code)
        ecols = get_emotion_cols(DATA[code]["train"])
        print(f"  ✓ {code.upper():<4} train={len(DATA[code]['train']):4d} "
              f"| val={len(DATA[code]['validation']):4d} | emotions={ecols}")
    except Exception as exc:
        print(f"  ERROR loading {code}: {exc}")

print("\nDataset loading complete.")


Loading BRIGHTER for all 9 languages ...
  ✓ ENG  train=2764 | val= 230 | emotions=['anger', 'fear', 'joy', 'sadness', 'surprise']
  ✓ HIN  train=2556 | val= 200 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ RUS  train=2679 | val= 398 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ HAU  train=2145 | val= 712 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ KIN  train=2451 | val= 814 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ SUN  train= 924 | val= 398 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ YOR  train=2992 | val= 994 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ VMW  train=1551 | val= 516 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  ✓ PCM  train=3728 | val=1240 | emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

Dataset loading complete.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluation and prediction utilities
# ─────────────────────────────────────────────────────────────────────────────

def labels_to_matrix(df: pd.DataFrame, emotion_cols: list) -> np.ndarray:
    """Extract label matrix, filling missing emotion columns with 0."""
    result = np.zeros((len(df), len(emotion_cols)), dtype=int)
    for i, e in enumerate(emotion_cols):
        if e in df.columns:
            result[:, i] = df[e].fillna(0).astype(int).values
    return result


ABSENT_EMOTIONS = {
    "eng": {"disgust"},
}


def get_eval_emotions(lang_code: str | None,
                      emotion_cols: list[str]) -> list[str]:
    if lang_code is None:
        return list(emotion_cols)

    absent = ABSENT_EMOTIONS.get(lang_code, set())
    return [emotion for emotion in emotion_cols if emotion not in absent]


def macro_f1(y_true: np.ndarray,
             y_pred: np.ndarray,
             emotion_cols: list[str],
             lang_code: str | None = None) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    if y_true.shape[1] != len(emotion_cols):
        raise ValueError(
            f"Expected {len(emotion_cols)} columns but received "
            f"{y_true.shape[1]}"
        )

    per_label = {}
    unrounded_scores = {}

    for index, label in enumerate(emotion_cols):
        score = f1_score(
            y_true[:, index],
            y_pred[:, index],
            zero_division=0,
        )
        unrounded_scores[label] = float(score)
        per_label[label] = round(float(score), 4)

    eval_emotions = get_eval_emotions(lang_code, emotion_cols)

    per_label["macro_f1"] = round(
        float(np.mean([unrounded_scores[e] for e in eval_emotions])),
        4,
    )

    return per_label


def save_predictions(phase, lang, condition, y_true, y_pred, emotions):
    payload = {"emotions": list(emotions),
               "y_true": np.asarray(y_true).astype(int).tolist(),
               "y_pred": np.asarray(y_pred).astype(int).tolist()}
    path = f"{PRED_DIR}/{phase}_{lang}_{condition}.json"
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(payload, f)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Validate prediction files and align their columns to EMOTION_ORDER.
# ─────────────────────────────────────────────────────────────────────────────

def _atomic_json_write_safe(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(tmp, path)


def load_align_prediction(path, rewrite=True):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    required = {"emotions", "y_true", "y_pred"}
    missing = required - set(payload)
    if missing:
        raise ValueError(f"{path}: missing keys {sorted(missing)}")
    emotions = list(payload["emotions"])
    if len(emotions) != len(set(emotions)):
        raise ValueError(f"{path}: duplicated emotion names {emotions}")
    if set(emotions) != set(EMOTION_ORDER):
        raise ValueError(f"{path}: expected {EMOTION_ORDER}, found {emotions}")
    y_true = np.asarray(payload["y_true"], dtype=int)
    y_pred = np.asarray(payload["y_pred"], dtype=int)
    if y_true.ndim != 2 or y_pred.ndim != 2 or y_true.shape != y_pred.shape:
        raise ValueError(f"{path}: incompatible arrays y_true={y_true.shape}, y_pred={y_pred.shape}")
    if y_true.shape[1] != len(emotions):
        raise ValueError(f"{path}: columns do not match stored emotion names")
    indices = [emotions.index(e) for e in EMOTION_ORDER]
    y_true = y_true[:, indices]
    y_pred = y_pred[:, indices]
    changed = emotions != EMOTION_ORDER
    if changed and rewrite:
        _atomic_json_write_safe(path, {
            "emotions": EMOTION_ORDER,
            "y_true": y_true.tolist(),
            "y_pred": y_pred.tolist(),
        })
        print(f"Reordered prediction: {os.path.basename(path)}")
    return y_true, y_pred, list(EMOTION_ORDER)


def score_saved_prediction(y_true, y_pred, lang_code):
    active = [e for e in EMOTION_ORDER if not (lang_code == "eng" and e == "disgust")]
    output = {}
    unrounded = {}
    for index, emotion in enumerate(EMOTION_ORDER):
        value = f1_score(y_true[:, index], y_pred[:, index], zero_division=0)
        unrounded[emotion] = float(value)
        output[emotion] = round(float(value), 4)
    output["macro_f1"] = round(float(np.mean([unrounded[e] for e in active])), 4)
    return output


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cultural prompt registry
# ─────────────────────────────────────────────────────────────────────────────
# P0: generic zero-shot (no cultural context)
# P1: cultural identity block
# P2: per-emotion cultural examples block
# P3: auto-built as P1 + P2 (truncated to 3000 chars) by build_cultural_block()
# ─────────────────────────────────────────────────────────────────────────────

CULTURAL_CONTENT = {

    # -- TIER 1 ------------------------------------------------------------------

    "eng": {
        "p1": """English is a low-context, individualist language where emotions are expressed directly and individually. The dominant cultural script values emotional self-control - managing displays is considered a sign of maturity. Positive emotions are expected openly in public; smiling toward strangers is a default social norm. Direct verbal labelling is the primary channel - English speakers name what they feel rather than conveying it indirectly. Individual emotional autonomy is prioritised over communal regulation. The Anglo cultural ideal emphasises managing emotion through rational self-direction rather than yielding to it or suppressing it entirely.""",

        "p2": """Joy: Publicly expressed and socially expected. Smiling toward strangers is normative. Happiness framed as individual achievement and personal wellbeing rather than communal experience. Sadness: Expressed directly through verbal statement. Prolonged public grief is socially uncomfortable - composure is expected relatively quickly. Sadness framed as temporary and manageable. Anger: Direct verbal expression culturally acceptable, particularly as assertiveness. Anger at injustice is legitimate. Aggressive physical displays are socially sanctioned against. Fear: Acknowledged verbally and directly. Self-disclosure of vulnerability is acceptable in appropriate contexts - less stigmatised than in high-context cultures. Surprise: Expressed openly with verbal exclamations. More restrained in formal contexts, more exuberant in casual speech. Disgust: Primarily sensory and physical rather than moral - triggered by contamination, bodily functions, and violations of physical purity. Maps to direct sensory repulsion more than moral transgression.""",
    },

    "hin": {
        "p1": """Hindi is a high-context, collectivist language where emotions are shaped by family honour, social hierarchy, and communal harmony. Indirect expression is the norm - conveyed through implication, silence, and bodily metaphors rather than direct statement. Izzat (honour) functions as a collective family asset; emotions threatening reputation are suppressed. Lajjā (modesty/shame) is a cultural virtue acting as public restraint, especially with elders. Collectivist display rules prioritise social harmony - negative emotions are masked to avoid disrupting group relations. Low-arousal states like contentment and peace are more valued than exuberant positivity. Emotions are expressed in Hindi-English code-mixed registers online.""",

        "p2": """Joy (khushi/sukh): Sukh is deep internal contentment without outward display. Happiness conceptualised through sweetness, illumination, and moral goodness. Tied to communal festivals not individual pleasure. Sadness (dukh/gham): Expressed through dukh (suffering), gham (grief), udaas (melancholy). Absorbed silently to protect family harmony. Anger (gussa/krodh): Gussa suppressed toward elders, acceptable toward subordinates. Krodh is intense destructive anger. Izzat violation is primary elicitor. Somaticised through reddening face and gnashing of teeth. Conceptualised as fluid, storm, or wild animal. Fear (darr/ghabrahat): Direct acknowledgement rare. Triggers: izzat violation and social judgement. Burden expressed through dil par patthar rakhna - placing a stone on the heart. Surprise (ashcharya): Less exuberant than English. Terms: ashcharya, adbhut. Exclamation arre marks sudden shift. Disgust (ghrina): No direct English equivalent. Maps to moral violation and ritual impurity rooted in Hindu purity-pollution norms, not sensory repulsion.""",
    },

    "rus": {
        "p1": """Russian emotional expression is heavily characterized by a cultural norm of stoicism in public spaces. Russians are culturally discouraged from displaying strong positive or negative emotions publicly, and actions like smiling at strangers are often considered insincere or inappropriate. In stark contrast to this public restraint, there is a deep cultural allowance for intense, uninhibited emotional expression within private, trusted relationships. The culture places significant value on enduring suffering, endurance, and melancholy, viewing these states as possessing an almost virtuous quality. This is encapsulated in the untranslatable concept of toska, which describes a profound longing or melancholy. While daily verbal expression may be restrained, the Russian literary and poetic tradition serves as a vibrant channel for emotions that are otherwise suppressed. Consequently, many complex emotional concepts exist primarily within this elevated literary register rather than in everyday speech. Furthermore, emotions are closely tied to physical embodiment, as linguistic collocations heavily treat the body as an organ of emotional expression. However, it is important to note that online emotional expression on platforms like VKontakte or Telegram has developed distinct, more disinhibited norms that diverge from traditional offline stoicism.""",

        "p2": """Joy is known as radost, but it is typically expressed privately or exclusively within close, trusted relationships. Communal celebration is reserved for specific occasions, as there is a widespread cultural suspicion of excessive or unearned public positivity. Sadness is conceptualised as grust' or the deeper toska, which represents a profound melancholy lacking a direct object. Enduring this sadness quietly is highly valued, reflecting the cultural virtue of suffering and resilience. Anger, termed gnev, is subject to strict public suppression but can reach extreme intensity in private settings. It becomes acceptable to display publicly only under specific conditions where the grievance is widely recognised as justified. Fear is termed strakh, and it is often expressed indirectly through dark humour and deflection rather than through direct, vulnerable acknowledgment. It may also be channelled through physical descriptions of the body's reaction rather than naming the emotion itself. Surprise is generally less effusive than its English equivalents. The linguistic markers used to indicate shock or surprise convey different levels of intensity and are applied more conservatively. Disgust is frequently framed in moral and aesthetic terms rather than purely physical revulsion. It is often expressed through an elevated literary or ironic register rather than via blunt, direct statements.""",
    },

    # -- TIER 2 ------------------------------------------------------------------

    "hau": {
        "p1": """Hausa emotional expression is heavily shaped by Islamic cultural influence and the core concept of kunya, which dictates a profound sense of shame and reserve in social interactions. This cultural framework requires emotional restraint, particularly in the presence of elders or those owed respect, ensuring that communal harmony is prioritized over individual outbursts. Another guiding cultural value is haƙuri, which prescribes patience, composure, and acceptance under distress or hardship. Consequently, Hausa speakers express emotions through a rich system of conventionalised indirect patterns rather than direct verbal statements. These indirect channels include deliberate silence, specific hand gestures, paralinguistic sounds, and the strategic use of proverbs to convey feelings without violating social norms. Furthermore, emotional language frequently relies on bodily metaphors, particularly using the word ciki, meaning stomach or heart area, to locate feeling within the physical body. Emotional display is also strictly moderated by gender, as what is considered acceptable expression differs significantly for men and women within the patriarchal structure. In addition, the deeply institutionalised belief in aljannu, or spirits, and tsafi, or witchcraft, functions as a powerful tool for social control and emotional regulation. Women, who face restrictive patriarchal structures, sometimes channel suppressed anger through indigenous prose fiction and metaphorical bodily expressions to safely voice dissent.""",

        "p2": """Joy is articulated as farin ciki, which translates literally to white stomach, framing happiness as a positive bodily state. Rather than through individual displays of pleasure, joy is expressed via communal celebrations, incorporating traditional songs and proverbs. Sadness is conceptualised as baƙin ciki, translating to black stomach. When dealing with grief, condolence greetings known as gaisuwar ta'aziya rely heavily on Islamic prayers and euphemisms, strictly avoiding direct verbal statements about death. Anger features multiple verbs of varying intensity, such as fusata, tunzura, harzua, and hasala, but direct expression is suppressed by the rules of kunya. Instead, anger is signalled through visual bodily cues like zare ido, meaning to pull the eye, or contempt idioms like sha kunu, meaning to drink gruel. Fear is strongly tied to the supernatural, particularly anxiety regarding aljannu and tsafi. The pervasive fear of spirit possession operates as a mechanism of social control, meaning fear is often expressed through religious or superstitious frameworks. Surprise is communicated through exclamatory sounds and facial mimicry rather than explicit verbal statements. This non-verbal approach remains consistent with the broader cultural norm of indirect emotional expression. Disgust is framed primarily as a moral and religious violation rather than a physical or sensory repulsion. Taboo violations that trigger disgust are sanctioned through social stigma and the threat of supernatural consequences.""",
    },

    "kin": {
        "p1": """The emotional landscape in Rwanda is fundamentally shaped by the post-genocide cultural context, where emotional restraint and silence are deeply sanctioned responses to trauma and distress. Rwanda possesses a documented culture of silence, meaning that strong emotions are expected to be processed internally rather than expressed in public spaces. Emotional management is largely governed by the concept of agaciro, which translates to dignity and self-worth, placing a premium on composure. Consequently, collective identity is prioritized over individual emotional display, making community solidarity the primary vessel for feelings. Communal rituals and structured environments, such as the gacaca justice proceedings, provide the sanctioned contexts for collective emotional expression. Vernacular memory practices and community solidarity create culturally specific avenues for mourning and processing shared history. Emotions tied directly to the collective trauma of the genocide do not map cleanly onto Western diagnostic or emotional categories. Thus, Rwandan emotional expression operates as a highly regulated system of communal processing and dignified restraint rather than a lack of feeling.""",

        "p2": """Joy is expressed through communal celebration framed heavily by the concept of agaciro and collective achievement. Individual displays of joy are expected to be modest, whereas communal joys during national events, church services, or family milestones are much more visible and acceptable. Sadness is normatively held internally as part of the broader culture of silence and dignified restraint. Public mourning is always communal and highly ritualised, relying on vernacular mourning practices through community memory rather than direct, individual verbal grief. Anger is publicly suppressed because overt individual aggression violates the sanctioned culture of silence and restraint. Instead, collective anger is processed formally through community mechanisms like the gacaca proceedings, maintaining a communal rather than an individual frame. Fear is prominently encapsulated by the term ihahamuka, a Rwanda-specific panic and fear response rooted in genocide trauma that lacks a Western equivalent. This concept combines feelings of fear, profound bodily distress, and collective memory into a unified expression. Surprise relies on restrained interpersonal cues rather than overt verbal exclamations. Exclamative structures using the question marker mbêga and manner noun ukūntu are characteristic markers, and the interjection yō signals sudden amazement. Disgust is expressed through social and moral framing tied to community reputation. Ishyano, meaning ritual impurity, and kuneena, the institutionalised avoidance of the morally contaminating, are primary mechanisms. The verb vugisha encodes the act of disgusting or sickening others.""",
    },

    "sun": {
        "p1": """Sundanese emotional expression is deeply rooted in the Austronesian cultural philosophy of Silih Asah, Silih Asih, and Silih Asuh, meaning mutual learning, mutual affection, and mutual care. This philosophical framework structures emotional expression as a fundamentally relational and communal experience rather than a purely individual one. Additionally, Islamic religious principles strongly influence emotional norms, demanding significant restraint, particularly regarding negative emotions like anger. In most interpersonal interactions, maintaining a flat facial expression serves as the dominant social cue to preserve harmony. Because overt facial displays are restricted, high vocal intonation and physical pointing gestures replace direct verbal emotional expression. Sundanese speakers also frequently utilize local cooperative frameworks like gotong royong to channel feelings into collective action. Pamali, the concept of taboo, and Islamic norms jointly regulate public emotional expression. The Lemes speech register cushions emotionally threatening communication and softens the expression of negative feelings. Consequently, research indicates that Sundanese speakers exhibit higher empathy orientations compared to other Indonesian ethnic groups.""",

        "p2": """Joy is expressed through communal celebrations and shared activities like the botram tradition that actively reflect the philosophy of Silih Asih. Individual joy is consistently framed in terms of relational harmony and community benefit rather than isolated personal pleasure. The concept of bodas, meaning white, encodes purity and happiness, and happiness is defined as virtuous living and inner calm rather than hedonic pleasure. Sadness is termed sedih or galau, and it is expressed indirectly through highly restrained body language and specific vocal intonations. The masking norm of crying in the heart while smiling on the face is culturally embedded. Grief is also channelled through traditional music, where madenda tuning evokes a sense of sacred melancholy. Anger is primarily non-verbal because direct verbal anger is socially discouraged and considered disruptive to relational harmony. Instead, individuals turn to Islamic coping responses such as wudhu for ritual washing, istighfar for seeking forgiveness, and the deliberate practice of patience. Fear is captured by the blended concepts of kawatir and takut, which merge anxiety and fear into a unified emotional state. Supernatural and spiritual concerns, heavily influenced by Islamic beliefs, remain prominent triggers for these fearful expressions. Surprise is uniquely marked by the exclamatory word meuni, meaning such or how, which frequently collocates with adjectives and interjections. The intensifier pisan amplifies affect, and the interjection Duh anchors climactic emotional moments. Disgust is expressed primarily through moral and religious purity framing, with najis, meaning ritually unclean, functioning as the primary trigger for jijik, the Sundanese term for disgust.""",
    },

    # -- TIER 3 ------------------------------------------------------------------

    "yor": {
        "p1": """Yoruba emotional expression is defined by a collectivist community orientation where feelings are rarely stated directly. Instead, emotions are channelled through an elaborate system of proverbs known as òwe, as well as through metaphors, folksongs, and oral poetry. Traditional festivals and communal gatherings provide the culturally sanctioned spaces for the collective processing of immense joy and grief. Outside of these communal rituals, individual emotional display is strictly moderated by powerful social norms regarding face, age, and respect. Direct confrontation is considered culturally inappropriate, meaning negative emotions are particularly subject to proverbial indirection. Proverbs function dynamically as both weapons of social power and diplomatic tools for conflict resolution. Furthermore, meaning is heavily supplemented by non-verbal semiotics, including specific hand gestures and facial expressions that carry exact cultural weight. Therefore, Yoruba emotional communication relies on a shared, highly contextual understanding of oral literature and bodily metaphor rather than explicit individual declaration.""",

        "p2": """Joy is expressed collectively through folksongs and traditional festivals that function as communal soul-menders. It is consistently framed in terms of community wellbeing, family harmony, and shared prosperity rather than isolated personal pleasure. Sadness is articulated indirectly through the recitation of proverbs, traditional dirges, and folksongs. The culture employs a strong somatic orientation, frequently using bodily organs like the heart metaphorically to locate and process sorrow. Anger relies heavily on proverbs acting as face-threatening acts, where powerful sayings function as indirect threats or insults to avoid discouraged direct confrontation. When described physically, anger utilizes heat and fire metaphors to illustrate how the emotion violently overwhelms the body. Fear is powerfully signalled non-verbally through the face of earnest, a culturally specific facial expression indicating grave seriousness that is easily misread cross-culturally. Additionally, speakers rely on describing sudden bodily sensations to locate fear rather than stating the emotion abstractly. Surprise is communicated via exclamatory interjections and abrupt shifts in body language. The symbolic placement of an ààlè, using objects like sand, leaves, or red cloth, conveys non-verbal warning and shock. Disgust is expressed through proverbs that explicitly invoke moral violation and the breach of social taboos. It represents a collective community judgement regarding disrespect for cultural norms rather than an individual sensory reaction.""",
    },

    "vmw": {
        "p1": """Emakhuwa emotional expression is profoundly shaped by its matrilineal Bantu social structure, which dictates distinct gender roles for processing feelings. Women hold a central role in communal ritual and the public expression of grief, whereas male gender norms strictly suppress vulnerable emotions like fear and sadness. Displaying grief loudly and publicly through ritualised mourning is viewed as a moral obligation to the community rather than a private, individual act. Conversely, collective joy is channelled through highly structured avenues like the tufo competitive dance and the Nakhula ancestral dance. Emotional management is also governed by the concept of ehaya, representing a form of shame tied intrinsically to communal reputation rather than individual guilt. Furthermore, collective fear and anger are frequently articulated through culturally specific idioms of sorcery that possess no direct Western equivalent. It is important to note that very little academic literature exists regarding Emakhuwa emotional expression in online contexts, so these offline anthropological norms remain the primary point of reference.""",

        "p2": """Joy is expressed collectively through events like the tufo competitive dance and Nakhula ancestral dance during harvests and marriages. Happiness is conceptualised as a collective communal experience tied to shared celebration and ancestral ritual. Sadness is expressed through ritualised, public wailing at funerals, which operates as a strict moral obligation. This loud, collective expression replaces quiet private grief, while indirect sadness is also processed through traditional dance and oral tradition. Anger is frequently expressed through sorcery idioms, utilizing terms like havara for leopard or sorcerer, and ekuluwe for pig to describe household discord. In broader political contexts, collective resistance and anger are signalled through phrases like anamalala, meaning it is over. Fear is heavily tied to sorcery and the mgosyo taboo system regarding hot and cold states. Anxiety concerning sorcerers among neighbours serves as the dominant expression of fear, while mgosyo transgressions produce a unique fear-guilt blend with physical bodily consequences. Surprise possesses very limited direct lexical expression in the language. Instead, sudden shock is conveyed through specific interjections and exaggerated body language rather than explicit vocabulary. Disgust is primarily a moral emotion associated with violations of mgosyo taboos and subsequent sorcery accusations. Physical or sensory disgust, as it is understood within Western frameworks, is significantly less prominent.""",
    },

    "pcm": {
        "p1": """Nigerian Pidgin, often referred to as Naijà, functions as a vital cross-ethnic lingua franca that bridges Nigeria's incredibly diverse cultural groups. Because it deliberately spans various communities, its emotional expression tends to be much more direct than indigenous languages like Yoruba or Hausa, while still maintaining its own distinct culturally Nigerian character. Corpus research indicates that negative sentiment is notably more prevalent in Nigerian language communities compared to other African language groups. A defining characteristic of the language is its heavy reliance on unique interjections, which serve as the most prominent and distinct emotional markers. Furthermore, emotional intensity is frequently conveyed through the grammatical process of reduplication, where repeating a word amplifies its feeling. Consistent with wider West African linguistic patterns, Nigerian Pidgin utilizes bodily sensation constructions that place emotion directly in the physical body rather than naming it abstractly. Online, this directness is further amplified, heavily mixing these culturally specific interjections with global social media norms and emojis.""",

        "p2": """Joy is frequently marked by the exclamatory interjection omo, which signals intense excitement, admiration, or positive shock. The language relies on communal celebratory phrasing, strongly favouring these distinctive Pidgin interjections over their standard English equivalents to express happiness. Sadness is expressed through bodily sensation constructions, most notably the phrase e pain me, which locates the sorrow as physical pain. Resigned sorrow or disbelief is captured by the marker na wa o, while online expressions frequently pair English interjections alongside sadness emojis. Anger is expressed very directly through phrases like I vex, meaning I am angry. Intensity is added through reduplication, such as saying vex vex, and anger frequently co-occurs with expressions of disgust within the exact same utterance. Fear is conveyed through phrases like I fear am, meaning I am afraid of it, grounding the feeling in bodily sensations rather than abstract names. Sharp, repeated pain or fearful distress is also expressed through sound-symbolic reduplication, such as the term CHUK CHUK. Surprise is marked by highly distinctive Pidgin interjections like chai and haba. Interestingly, these shocked expressions tend to occur much more frequently in contexts of negative surprise than in positive ones. Disgust is widely expressed using the direct phrase e no good, meaning it is not good. This relies heavily on a dominant moral framing and frequently clusters together with anger in the same sentence.""",
    },
}


def build_cultural_block(lang_code: str, prompt_key: str) -> str:
    """
    Build the cultural_block string for a given lang/prompt.
    P3 = P1 + newline + P2, truncated to 3200 chars.
    P0 returns empty string (no cultural block).
    """
    if prompt_key == "p0":
        return ""
    content = CULTURAL_CONTENT[lang_code]
    if prompt_key == "p1":
        return content["p1"].strip()
    if prompt_key == "p2":
        return content["p2"].strip()
    if prompt_key == "p3":
        combined = content["p1"].strip() + "\n\n" + content["p2"].strip()
        return combined[:3200]
    raise ValueError(f"Unknown prompt_key: {prompt_key}")


def build_prompt(lang_code: str, prompt_key: str, text: str) -> str:
    """Build the canonical P0–P3 Gemma prompt for one test item."""
    GENERIC_SYSTEM = (
        "You are an emotion classification system.\n"
        "Read the text below and identify which emotions it expresses.\n"
        "Available emotions: joy, sadness, anger, fear, surprise, disgust\n"
        "A text may express multiple emotions, one emotion, or none at all.\n\n"
        "Instructions:\n"
        "- Respond ONLY with a comma-separated list of emotion labels from the list above.\n"
        "- Use lowercase exactly as written above.\n"
        "- If no emotion is present, respond with the single word: neutral\n"
        "- Do not add any explanation, punctuation, or extra text."
    )

    cultural_block = build_cultural_block(lang_code, prompt_key)

    if cultural_block:
        system_content = (
            "You are a culturally-aware emotion classification system.\n\n"
            + cultural_block
            + "\n\n"
            "Read the text below and identify which emotions it expresses.\n"
            "Available emotions: joy, sadness, anger, fear, surprise, disgust\n"
            "A text may express multiple emotions, one emotion, or none at all.\n\n"
            "Instructions:\n"
            "- Respond ONLY with a comma-separated list of emotion labels from the list above.\n"
            "- Use lowercase exactly as written above.\n"
            "- If no emotion is present, respond with the single word: neutral\n"
            "- Do not add any explanation, punctuation, or extra text."
        )
    else:
        system_content = GENERIC_SYSTEM

    messages = [
        {"role": "user", "content": f"{system_content}\n\nText: {text}"}
    ]

    prompt = _try_apply_chat_template(messages)
    return prompt + "Emotions:"


print("Cultural prompt registry loaded.")
print(f"Languages: {list(CULTURAL_CONTENT)}")
print(f"Prompt conditions: {[key.upper() for key in PROMPT_KEYS]}")


Cultural prompt registry loaded.
Languages: ['eng', 'hin', 'rus', 'hau', 'kin', 'sun', 'yor', 'vmw', 'pcm']
Prompt conditions: ['P0', 'P1', 'P2', 'P3']


In [ ]:
# Validate the cultural-context lengths used by the four prompt conditions.

print("P1 lengths:")
for code in LANG_CODES:
    block = CULTURAL_CONTENT[code]["p1"].strip()
    status = "OK" if len(block) <= 1800 else "OVER LIMIT"
    print(f"  {code.upper()}: {len(block)} chars — {status}")

print("\nP2 lengths:")
for code in LANG_CODES:
    block = CULTURAL_CONTENT[code]["p2"].strip()
    status = "OK" if len(block) <= 1800 else "OVER LIMIT"
    print(f"  {code.upper()}: {len(block)} chars — {status}")

print("\nP3 lengths (P1+P2 combined, limit=3200):")
for code in LANG_CODES:
    block = build_cultural_block(code, "p3")
    raw_len = len(CULTURAL_CONTENT[code]["p1"].strip()) + 2 + len(CULTURAL_CONTENT[code]["p2"].strip())
    truncated = raw_len > 3200
    status = "TRUNCATED" if truncated else "OK"
    print(f"  {code.upper()}: {len(block)} chars — {status}")


P1 lengths:
  ENG: 651 chars — OK
  HIN: 731 chars — OK
  RUS: 1361 chars — OK
  HAU: 1530 chars — OK
  KIN: 1208 chars — OK
  SUN: 1266 chars — OK
  YOR: 1139 chars — OK
  VMW: 1141 chars — OK
  PCM: 1174 chars — OK

P2 lengths:
  ENG: 1047 chars — OK
  HIN: 1056 chars — OK
  RUS: 1405 chars — OK
  HAU: 1527 chars — OK
  KIN: 1651 chars — OK
  SUN: 1758 chars — OK
  YOR: 1554 chars — OK
  VMW: 1505 chars — OK
  PCM: 1518 chars — OK

P3 lengths (P1+P2 combined, limit=3200):
  ENG: 1700 chars — OK
  HIN: 1789 chars — OK
  RUS: 2768 chars — OK
  HAU: 3059 chars — OK
  KIN: 2861 chars — OK
  SUN: 3026 chars — OK
  YOR: 2695 chars — OK
  VMW: 2648 chars — OK
  PCM: 2694 chars — OK


In [ ]:
# Load the model only when inference is explicitly requested.
if RUN_MODEL_INFERENCE:
    login(token=HF_TOKEN)

    # ─────────────────────────────────────────────────────────────────────────────
    # Load Gemma 4
    # ─────────────────────────────────────────────────────────────────────────────

    USE_MULTIMODAL_LOADER = True

    os.environ["HF_HOME"] = LOCAL_MODEL_CACHE
    os.environ["TRANSFORMERS_CACHE"] = f"{LOCAL_MODEL_CACHE}/hub"
    os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

    try:
        gemma_model
        gemma_tok
        print("Gemma-4 already loaded this session -- skipping reload")
    except NameError:
        print(f"Loading {GEMMA_HF_NAME} ...")
        print("NOTE: Requires A100 40GB+.")

        if USE_MULTIMODAL_LOADER:
            from transformers import AutoProcessor, AutoModelForMultimodalLM
            gemma_processor = AutoProcessor.from_pretrained(
                GEMMA_HF_NAME, token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE,
            )
            if LOAD_IN_4BIT_ON_THE_FLY:
                bnb_cfg = BitsAndBytesConfig(
                    load_in_4bit              = True,
                    bnb_4bit_compute_dtype    = torch.bfloat16,
                    bnb_4bit_use_double_quant = True,
                    bnb_4bit_quant_type       = "nf4",
                )
                gemma_model = AutoModelForMultimodalLM.from_pretrained(
                    GEMMA_HF_NAME, quantization_config=bnb_cfg, device_map="auto",
                    attn_implementation="sdpa", token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE,
                )
            else:
                gemma_model = AutoModelForMultimodalLM.from_pretrained(
                    GEMMA_HF_NAME, dtype="auto", device_map="auto",
                    attn_implementation="sdpa", token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE,
                )
            gemma_tok = gemma_processor
        else:
            if LOAD_IN_4BIT_ON_THE_FLY:
                bnb_cfg = BitsAndBytesConfig(
                    load_in_4bit              = True,
                    bnb_4bit_compute_dtype    = torch.float16,
                    bnb_4bit_use_double_quant = True,
                    bnb_4bit_quant_type       = "nf4",
                )
                gemma_model = AutoModelForCausalLM.from_pretrained(
                    GEMMA_HF_NAME,
                    quantization_config = bnb_cfg,
                    device_map           = "auto",
                    attn_implementation  = "sdpa",
                    token                = HF_TOKEN,
                    cache_dir=LOCAL_MODEL_CACHE,
                )
            else:
                gemma_model = AutoModelForCausalLM.from_pretrained(
                    GEMMA_HF_NAME,
                    device_map           = "auto",
                    attn_implementation  = "sdpa",
                    token                = HF_TOKEN,
                    cache_dir=LOCAL_MODEL_CACHE,
                )

            gemma_tok = AutoTokenizer.from_pretrained(
                GEMMA_HF_NAME,
                token     = HF_TOKEN,
                cache_dir=LOCAL_MODEL_CACHE,
            )

    gemma_model.eval()

    # ── Resolve tokenizer-level attrs regardless of loader branch ────────────────
    # Multimodal branch: gemma_tok is a Processor -> tokenizer-level attrs live
    # under .tokenizer. CausalLM branch: gemma_tok IS the tokenizer already.
    _inner_tok = getattr(gemma_tok, "tokenizer", gemma_tok)

    THINKING_KWARG_NAME = "enable_thinking"

    def _try_apply_chat_template(messages):
        """Attempt apply_chat_template with the thinking-disable kwarg, falling
        back cleanly to no kwarg if it's rejected. Verified pattern from Phase 1:
        the kwarg isn't a named parameter in the signature but IS accepted via a
        catch-all **kwargs without raising."""
        kwargs = dict(tokenize=False, add_generation_prompt=True)
        kwargs[THINKING_KWARG_NAME] = False
        try:
            return gemma_tok.apply_chat_template(messages, **kwargs)
        except TypeError:
            kwargs.pop(THINKING_KWARG_NAME)
            return gemma_tok.apply_chat_template(messages, **kwargs)

    GENERATION_CONFIG["pad_token_id"] = _inner_tok.eos_token_id

    # ── VRAM check ────────────────────────────────────────────────────────────────
    vram_used  = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"v {GEMMA_HF_NAME} loaded.")
    print(f"  VRAM used : {vram_used:.1f} GB / {vram_total:.1f} GB total")
    print(f"  eos_token_id resolved to: {_inner_tok.eos_token_id}")
    if vram_used > vram_total * 0.9:
        print("  WARNING: VRAM >90% — high OOM risk. Reduce batch_size in Cell 7.")

else:
    print("Inference disabled; existing Phase 2 predictions will be validated.")


Inference disabled; existing Phase 2 predictions will be validated.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inference engine
# ─────────────────────────────────────────────────────────────────────────────

def _atomic_json_write(path: str, data) -> None:
    """Write JSON atomically: temp file + os.replace(). A disconnect mid-write
    leaves the OLD file intact (or no file at all on first write), never a
    truncated/corrupted one."""
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w") as f:
        json.dump(data, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)


def run_gemma4_px(lang_code: str, prompt_key: str,
                  batch_size: int = 4) -> dict:
    """
    Run Gemma 4 with a given prompt variant (p0/p1/p2/p3) on the validation/dev split.\n    P0 is the single canonical generic baseline reused by Phase 1 and Phase 4.
    Stores six emotion columns; English macro-F1 is calculated over
    five active emotions because disgust is structurally absent.
    Track A and Track C share the same output (no training data used).
    Checkpoints every 50 samples to Drive — safe to interrupt and restart.
    Writes are atomic (see _atomic_json_write).

    Args:
        lang_code:  e.g. "yor"
        prompt_key: "p0" | "p1" | "p2" | "p3"
        batch_size: Number of prompts generated in each inference batch.
    Returns:
        dict with per-emotion F1 + macro_f1
    """
    df     = DATA[lang_code]["validation"].copy()
    y_true = labels_to_matrix(df, EMOTION_ORDER)
    y_pred = np.zeros_like(y_true)
    texts  = df["text"].tolist()

    # ── Resume from checkpoint  ─────────
    ckpt_path = f"{P2_SAVE_DIR}/{lang_code}_{prompt_key}_progress.json"
    start_idx = 0

    if os.path.exists(ckpt_path):
        try:
            with open(ckpt_path) as f:
                ckpt = json.load(f)
            y_pred    = np.array(ckpt["y_pred"])
            start_idx = ckpt["processed"]
            print(f"    Resuming {lang_code}/{prompt_key} "
                  f"from {start_idx}/{len(texts)}")
        except (json.JSONDecodeError, KeyError, ValueError) as exc:
            print(f"    WARNING: checkpoint for {lang_code}/{prompt_key} "
                  f"is corrupted/unreadable ({exc}).")
            print(f"    Restarting {lang_code}/{prompt_key} from sample 0.")
            y_pred    = np.zeros_like(y_true)
            start_idx = 0

    # ── Inference loop ───────────────────────────────────────────────────────
    for batch_start in range(start_idx, len(texts), batch_size):
        batch_texts = texts[batch_start : batch_start + batch_size]
        prompts     = [build_prompt(lang_code, prompt_key, t)
                       for t in batch_texts]

        # Left-padding required for batched generation with decoder-only models.
        _inner_tok.padding_side = "left"
        inputs = gemma_tok(
            text           = prompts,   # MUST be keyword - Gemma4Processor takes images positionally first
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 2048,
        ).to(gemma_model.device)

        with torch.no_grad():
            output_ids = gemma_model.generate(**inputs, **GENERATION_CONFIG)

        for i, out in enumerate(output_ids):
            # Slice off input tokens - keep only newly generated tokens
            new_ids  = out[inputs["input_ids"].shape[1]:]
            raw      = gemma_tok.decode(
                new_ids, skip_special_tokens=True).strip().lower()
            detected = (
                []
                if raw in ("neutral", "")
                else [e.strip() for e in raw.split(",")
                      if e.strip() in EMOTION_ORDER]
            )
            for j, lbl in enumerate(EMOTION_ORDER):
                if lbl in detected:
                    y_pred[batch_start + i, j] = 1

        processed = min(batch_start + batch_size, len(texts))

        # ── Checkpoint every 50 samples  ───────────────────────
        if processed % 50 == 0 or processed == len(texts):
            _atomic_json_write(ckpt_path, {"y_pred": y_pred.tolist(), "processed": processed})
            print(f"      {processed}/{len(texts)} samples ... (checkpoint saved)")

    # ── Clean up checkpoint on success ───────────────────────────────────────
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
        print(f"    Checkpoint cleared for {lang_code}/{prompt_key}")

    save_predictions("validation", lang_code, f"prompt_{prompt_key}",
                     y_true, y_pred, EMOTION_ORDER)
    return macro_f1(y_true, y_pred, EMOTION_ORDER, lang_code=lang_code)


if RUN_MODEL_INFERENCE:
    print("Gemma inference engine ready.")
else:
    print("Inference is disabled; saved predictions will be validated.")


Inference is disabled; saved predictions will be validated.


In [ ]:
# Execute or validate the 36 Phase 2 conditions
RESULTS_PATH = f"{P2_SAVE_DIR}/phase2_validation_raw_results_gemma4.json"

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r", encoding="utf-8") as handle:
        phase2_results = json.load(handle)
else:
    phase2_results = {}


def validate_phase2_prediction(code, prompt_key):
    path = f"{PRED_DIR}/validation_{code}_prompt_{prompt_key}.json"
    if not os.path.exists(path):
        return False, None, f"missing prediction file: {path}"
    try:
        y_true, y_pred, emotions = load_align_prediction(path, rewrite=True)
    except (OSError, json.JSONDecodeError, ValueError) as exc:
        return False, None, str(exc)

    expected_y_true = labels_to_matrix(DATA[code]["validation"], EMOTION_ORDER)
    if y_true.shape != expected_y_true.shape:
        return False, None, (
            f"gold shape {y_true.shape} != expected {expected_y_true.shape}"
        )
    if not np.array_equal(y_true, expected_y_true):
        return False, None, "gold labels or test-row order do not match"

    payload = {
        "emotions": emotions,
        "y_true": y_true,
        "y_pred": y_pred,
    }
    return True, payload, ""


invalid_conditions = []
for code in LANG_ORDER:
    phase2_results.setdefault(code, {})
    for prompt_key in PROMPT_KEYS:
        valid, payload, reason = validate_phase2_prediction(code, prompt_key)
        if not valid:
            phase2_results[code].pop(prompt_key, None)
            invalid_conditions.append((code, prompt_key, reason))
            continue

        phase2_results[code][prompt_key] = macro_f1(
            payload["y_true"],
            payload["y_pred"],
            EMOTION_ORDER,
            lang_code=code,
        )

_atomic_json_write_safe(RESULTS_PATH, phase2_results)

if invalid_conditions and not RUN_MODEL_INFERENCE:
    details = "\n".join(
        f"  {code}/{prompt}: {reason}"
        for code, prompt, reason in invalid_conditions
    )
    raise RuntimeError(
        "Phase 2 predictions are incomplete or invalid. "
        "Set RUN_MODEL_INFERENCE=True to generate them:\n" + details
    )

if RUN_MODEL_INFERENCE:
    for code in LANG_ORDER:
        for prompt_key in PROMPT_KEYS:
            valid, _, _ = validate_phase2_prediction(code, prompt_key)
            if valid:
                print(
                    f"Skipping {code.upper()}/{prompt_key.upper()}: "
                    "valid prediction already exists."
                )
                continue

            print(f"Running {code.upper()}/{prompt_key.upper()}...")
            result = run_gemma4_px(code, prompt_key, batch_size=4)
            phase2_results[code][prompt_key] = result
            _atomic_json_write_safe(RESULTS_PATH, phase2_results)
            print(f"macro-F1: {result['macro_f1']:.4f}")

    if "gemma_model" in globals():
        del gemma_model
        torch.cuda.empty_cache()

expected_conditions = len(LANG_ORDER) * len(PROMPT_KEYS)
completed_conditions = sum(
    prompt in phase2_results.get(code, {})
    for code in LANG_ORDER
    for prompt in PROMPT_KEYS
)
if completed_conditions != expected_conditions:
    raise AssertionError(
        f"Expected {expected_conditions} completed conditions, "
        f"found {completed_conditions}."
    )

print(f"Phase 2 complete: {completed_conditions} conditions verified.")


Phase 2 complete: 36 conditions verified.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Results and exports
# ─────────────────────────────────────────────────────────────────────────────

def build_phase2_summary() -> pd.DataFrame:
    rows = []
    for code in LANG_ORDER:
        info = LANGUAGES[code]
        if code not in phase2_results:
            continue
        row = {
            "Language":          info["name"],
            "Language code":     code.upper(),
            "Resource tier":     info["tier"],
            "Linguistic subgroup": info["subfamily"],
        }
        for pk in PROMPT_KEYS:
            col = PROMPT_LABELS[pk]
            row[col] = phase2_results[code].get(pk, {}).get("macro_f1", None)

        available = {pk: phase2_results[code][pk]["macro_f1"]
                     for pk in PROMPT_KEYS if pk in phase2_results[code]}
        if available:
            best_key                          = max(available, key=available.get)
            row["Best validation prompt"]     = best_key.upper()
            row["Best validation macro-F1"]   = available[best_key]
            if "p0" in available:
                row["Improvement over P0 (best minus P0)"] = round(available[best_key] - available["p0"], 4)
        row["Evaluation split"] = "Validation"
        row["Analysis scope"]   = PHASE2_SCOPE
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


print("=" * 70)
print("  PHASE 2 RESULTS — Cultural Prompt Ablation (Gemma 4)")
print("=" * 70)
summary = build_phase2_summary()
print(summary.to_string(index=False))

# ── Save outputs ──────────────────────────────────────────────────────────────
summary_path = f"{P2_SAVE_DIR}/phase2_gemma4_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"\nSummary saved to {summary_path}")

# Per-prompt detail table
detail_rows = []
for code in LANG_ORDER:
    info = LANGUAGES[code]
    if code not in phase2_results:
        continue
    for pk in PROMPT_KEYS:
        if pk not in phase2_results[code]:
            continue
        res = phase2_results[code][pk]
        row = {
            "Language":        info["name"],
            "Language code":   code.upper(),
            "Resource tier":   info["tier"],
            "Prompt condition": pk.upper(),
            "Anger F1":        res.get("anger"),
            "Disgust F1":      res.get("disgust"),
            "Fear F1":         res.get("fear"),
            "Joy F1":          res.get("joy"),
            "Sadness F1":      res.get("sadness"),
            "Surprise F1":     res.get("surprise"),
            "Macro-F1":        res.get("macro_f1"),
            "Evaluation split": "Validation",
            "Analysis scope":   PHASE2_SCOPE,
        }
        detail_rows.append(row)

detail_df   = pd.DataFrame(detail_rows)
detail_path = f"{P2_SAVE_DIR}/phase2_gemma4_detailed.csv"
detail_df.to_csv(detail_path, index=False)
print(f"Detail saved to {detail_path}")
print(f"Raw JSON at {RESULTS_PATH}")

# Canonical P0 audit manifest used by downstream notebooks.
canonical_manifest = {
    "source_file": RESULTS_PATH,
    "condition": CANONICAL_P0_KEY,
    "reused_as": [
        "Phase 1 B3 Gemma baseline",
        "Phase 2 control condition",
        "Phase 4 neither condition",
    ],
    "language_order": LANG_ORDER,
    "emotion_order": EMOTION_ORDER,
}

manifest_path = f"{P2_SAVE_DIR}/canonical_p0_manifest.json"
_atomic_json_write(manifest_path, canonical_manifest)
print(f"Canonical P0 manifest saved to {manifest_path}")


  PHASE 2 RESULTS — Cultural Prompt Ablation (Gemma 4)
       Language Language code  Resource tier Linguistic subgroup  P0 macro-F1 (basic prompt)  P1 macro-F1 (cultural identity context)  P2 macro-F1 (cultural examples)  P3 macro-F1 (combined cultural context) Best validation prompt  Best validation macro-F1  Improvement over P0 (best minus P0) Evaluation split                   Analysis scope
        English           ENG              1            Germanic                      0.6274                                   0.6112                           0.6069                                   0.6045                     P0                    0.6274                               0.0000       Validation Internal Gemma prompt comparison
          Hindi           HIN              1          Indo-Aryan                      0.7421                                   0.7715                           0.7719                                   0.7752                     P3                    0.7752 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Phase 2 Bootstrap Significance
# ─────────────────────────────────────────────────────────────────────────────

N_BOOTSTRAP  = 10_000
_BOOT_SEED   = 42
_BOOT_RNG    = np.random.default_rng(_BOOT_SEED)
SIG_OUT_PATH = f"{P2_SAVE_DIR}/phase2_bootstrap_significance.csv"


def _macro_f1_active_np(yt_active, yp_active):
    """
    Macro-F1 over a 2-D (n_samples, n_active_emotions) pair of int arrays.
    Handles the zero-denominator case per column.
    """
    scores = []
    for j in range(yt_active.shape[1]):
        tp    = float(np.sum(yt_active[:, j] * yp_active[:, j]))
        fp    = float(np.sum((1 - yt_active[:, j]) * yp_active[:, j]))
        fn    = float(np.sum(yt_active[:, j] * (1 - yp_active[:, j])))
        denom = 2 * tp + fp + fn
        scores.append(2 * tp / denom if denom > 0 else 0.0)
    return float(np.mean(scores))


def paired_bootstrap_p2(y_true, y_pred_a, y_pred_b, lang_code,
                         emotions=EMOTION_ORDER, n=10_000, rng=None):
    """
    Fully-vectorised paired bootstrap for macro-F1 difference (b vs a).
    All n resamples are computed simultaneously per emotion column using
    numpy fancy indexing, so this runs in seconds not minutes.

    Returns: (delta, ci_low, ci_high, p_value, f1_a, f1_b)
    p_value is two-sided and floored at 1/n (never exactly 0).
    """
    if rng is None:
        rng = np.random.default_rng(42)

    # Active emotions (English excludes disgust)
    absent     = ABSENT_EMOTIONS.get(lang_code, set())
    active_idx = np.array([i for i, e in enumerate(emotions) if e not in absent],
                           dtype=np.intp)
    n_active   = len(active_idx)
    n_samples  = y_true.shape[0]

    # Slice to active emotions, int32
    yt = y_true[:, active_idx].astype(np.int32)      # (n_samples, n_active)
    ya = y_pred_a[:, active_idx].astype(np.int32)
    yb = y_pred_b[:, active_idx].astype(np.int32)

    # Observed macro-F1 for both systems
    f1_a = _macro_f1_active_np(yt, ya)
    f1_b = _macro_f1_active_np(yt, yb)
    observed_delta = f1_b - f1_a

    # Draw all resample indices at once: (n, n_samples)
    all_idx = rng.integers(0, n_samples, size=(n, n_samples))

    # Per-emotion bootstrap F1 matrices: (n, n_active)
    boot_f1_a = np.zeros((n, n_active))
    boot_f1_b = np.zeros((n, n_active))

    for col in range(n_active):
        yt_col  = yt[:, col]
        ya_col  = ya[:, col]
        yb_col  = yb[:, col]

        # Fancy index: each row of all_idx selects one resample
        yt_boot = yt_col[all_idx]     # (n, n_samples)
        ya_boot = ya_col[all_idx]
        yb_boot = yb_col[all_idx]

        tp_a    = np.sum(yt_boot * ya_boot, axis=1, dtype=np.float64)  # (n,)
        denom_a = (2 * tp_a
                   + np.sum((1 - yt_boot) * ya_boot,       axis=1, dtype=np.float64)
                   + np.sum(yt_boot       * (1 - ya_boot), axis=1, dtype=np.float64))
        boot_f1_a[:, col] = np.where(denom_a > 0, 2 * tp_a / denom_a, 0.0)

        tp_b    = np.sum(yt_boot * yb_boot, axis=1, dtype=np.float64)
        denom_b = (2 * tp_b
                   + np.sum((1 - yt_boot) * yb_boot,       axis=1, dtype=np.float64)
                   + np.sum(yt_boot       * (1 - yb_boot), axis=1, dtype=np.float64))
        boot_f1_b[:, col] = np.where(denom_b > 0, 2 * tp_b / denom_b, 0.0)

    # Macro-F1 per resample: mean over emotions, then delta
    boot_deltas = boot_f1_b.mean(axis=1) - boot_f1_a.mean(axis=1)   # (n,)

    ci_low  = float(np.percentile(boot_deltas, 2.5))
    ci_high = float(np.percentile(boot_deltas, 97.5))

    boot_deltas_shifted = boot_deltas - observed_delta
    count   = int(np.sum(np.abs(boot_deltas_shifted) >= abs(observed_delta)))
    p_value = max(count / n, 1.0 / n)   # floor at 1/n; never exactly 0

    return float(observed_delta), ci_low, ci_high, p_value, f1_a, f1_b


def holm_correct_p2(raw_p_values):
    """Holm-Bonferroni correction; preserves original order of p-values."""
    n     = len(raw_p_values)
    order = sorted(range(n), key=lambda i: raw_p_values[i])
    adj   = [0.0] * n
    floor = 0.0
    for rank, idx in enumerate(order):
        a     = min(raw_p_values[idx] * (n - rank), 1.0)
        a     = max(a, floor)
        floor = a
        adj[idx] = a
    return adj


# ─────────────────────────────────────────────────────────────────────────────
print(f"Paired bootstrap: {N_BOOTSTRAP:,} resamples | seed={_BOOT_SEED}")
print(f"Prediction files: {PRED_DIR}")
print(f"Output: {SIG_OUT_PATH}\n")

all_sig_rows = []

for code in LANG_ORDER:
    lang_name = LANGUAGES[code]["name"]
    p0_path   = f"{PRED_DIR}/validation_{code}_prompt_p0.json"

    if not os.path.exists(p0_path):
        print(f"[{code.upper()}] SKIP  P0 prediction file not found")
        print(f"  Expected: {p0_path}")
        continue

    try:
        y_true, y_pred_p0, _ = load_align_prediction(p0_path, rewrite=False)
    except Exception as exc:
        print(f"[{code.upper()}] SKIP  could not load P0 predictions: {exc}")
        continue

    lang_rows   = []
    lang_p_vals = []

    for px in ["p1", "p2", "p3"]:
        px_path = f"{PRED_DIR}/validation_{code}_prompt_{px}.json"
        if not os.path.exists(px_path):
            print(f"  [{code.upper()}/{px.upper()}] SKIP  prediction file not found")
            continue
        try:
            _, y_pred_px, _ = load_align_prediction(px_path, rewrite=False)
        except Exception as exc:
            print(f"  [{code.upper()}/{px.upper()}] SKIP  {exc}")
            continue

        delta, ci_low, ci_high, p_val, f1_p0, f1_px = paired_bootstrap_p2(
            y_true, y_pred_p0, y_pred_px, code,
            emotions=EMOTION_ORDER, n=N_BOOTSTRAP, rng=_BOOT_RNG,
        )

        lang_rows.append({
            "Language code":                              code.upper(),
            "Language":                                   lang_name,
            "Prompt condition":                           px,
            "Comparison":                                 f"P0 vs {px.upper()}",
            "P0 baseline macro-F1":                       round(f1_p0, 4),
            "Compared prompt macro-F1":                   round(f1_px, 4),
            "Prompt effect (compared prompt minus P0)":   round(delta, 4),
            "95% confidence interval lower bound":        round(ci_low, 4),
            "95% confidence interval upper bound":        round(ci_high, 4),
            "Raw p-value":                                round(p_val, 6),
            "Higher-scoring condition":                   px.upper() if delta > 0 else "P0",
        })
        lang_p_vals.append(p_val)
        print(f"  [{code.upper()}/{px.upper()}]  "
              f"P0={f1_p0:.4f}  {px.upper()}={f1_px:.4f}  "
              f"delta={delta:+.4f}  CI=[{ci_low:+.4f},{ci_high:+.4f}]  "
              f"p={p_val:.4f}")

    # Holm correction within this language's 3 comparisons
    if lang_rows:
        adj = holm_correct_p2(lang_p_vals)
        for row, a in zip(lang_rows, adj):
            row["Holm-adjusted p-value"]                       = round(a, 6)
            row["Significant after Holm correction (alpha=0.05)"] = bool(a < 0.05)
            row["Evaluation split"]                            = "Validation"
            row["Analysis scope"]                              = PHASE2_SCOPE
        all_sig_rows.extend(lang_rows)
    print()

sig_df = pd.DataFrame(all_sig_rows)

if sig_df.empty:
    print("WARNING: No rows produced.")
    print("Check that prediction files exist in PRED_DIR before running this cell.")
else:
    # Atomic write to Drive
    _tmp = f"{SIG_OUT_PATH}.tmp"
    sig_df.to_csv(_tmp, index=False)
    os.replace(_tmp, SIG_OUT_PATH)
    print(f"Saved {len(sig_df)} rows  →  {SIG_OUT_PATH}")
    n_sig = int(sig_df["Significant after Holm correction (alpha=0.05)"].sum())
    print(f"Significant (Holm p<0.05): {n_sig}/{len(sig_df)}")
    print()
    print(sig_df.to_string(index=False))


Paired bootstrap: 10,000 resamples | seed=42
Prediction files: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase2/predictions
Output: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase2/phase2_bootstrap_significance.csv

  [ENG/P1]  P0=0.6274  P1=0.6112  delta=-0.0162  CI=[-0.0363,+0.0028]  p=0.1029
  [ENG/P2]  P0=0.6274  P2=0.6069  delta=-0.0206  CI=[-0.0441,+0.0028]  p=0.0845
  [ENG/P3]  P0=0.6274  P3=0.6045  delta=-0.0229  CI=[-0.0445,-0.0013]  p=0.0373

  [HIN/P1]  P0=0.7421  P1=0.7715  delta=+0.0294  CI=[+0.0067,+0.0547]  p=0.0169
  [HIN/P2]  P0=0.7421  P2=0.7719  delta=+0.0298  CI=[-0.0073,+0.0659]  p=0.1053
  [HIN/P3]  P0=0.7421  P3=0.7752  delta=+0.0331  CI=[-0.0054,+0.0701]  p=0.0830

  [RUS/P1]  P0=0.8180  P1=0.8414  delta=+0.0233  CI=[+0.0068,+0.0406]  p=0.0065
  [RUS/P2]  P0=0.8180  P2=0.8272  delta=+0.0092  CI=[-0.0132,+0.0305]  p=0.4187
  [RUS/P3]  P0=0.8180  P3=0.8412  delta=+0.0231  CI=[+0.0026,+0.0439]  p=0.0289

  [HAU/P1]  P0=0.5606  P1=0.606